# ⚽ CDAF: Exploratory Data Analysis (EDA)
**Dataset:** Bundesliga 24/25 - CazéTV Live Chat

Este notebook realiza uma análise estatística e visual completa de todo o corpus de mensagens coletadas, seguindo as melhores práticas de Data Science.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import glob
import numpy as np

# Configuração Global
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 12

def t2s(t): 
    if not t: return 0
    parts = list(map(int, t.split(':')))
    return parts[0]*3600 + parts[1]*60 + parts[2] if len(parts)==3 else parts[0]*60 + parts[1]

## 1. Data Ingestion & Consolidation
Carregamos todos os 66 arquivos CSV processados em um único DataFrame mestre.

In [ ]:
processed_files = glob.glob('data/processed/chat_*_processed.csv')
dfs = []

for f in processed_files:
    video_id = os.path.basename(f).split('_')[1]
    temp_df = pd.read_csv(f)
    temp_df['video_id'] = video_id
    dfs.append(temp_df)

master_df = pd.concat(dfs, ignore_index=True)

# Conversão de tipos para eficiência
master_df['periodo'] = master_df['periodo'].astype('category')
master_df['minuto_arredondado'] = master_df['timestamp_jogo_segundos'] // 60

print(f"Dataset consolidado com sucesso!")
print(f"Total de mensagens: {len(master_df):,}")
print(f"Total de partidas (arquivos): {len(processed_files)}")
master_df.head()

## 2. Estatísticas de Alto Nível
Entendendo o volume e a abrangência dos dados.

In [ ]:
summary = {
    "Total Mensagens": len(master_df),
    "Autores Únicos": master_df['autor'].nunique(),
    "Média Mensagens por Jogo": len(master_df) / len(processed_files),
    "Mensagem mais comum": str(master_df['mensagem'].mode()[0]),
    "Frequência do Top 1": master_df['mensagem'].value_counts().iloc[0]
}

pd.Series(summary).to_frame(name="Valor")

## 3. Análise Temporal (Engajamento na Temporada)
Volume médio de mensagens por minuto em todas as partidas.

In [ ]:
# Agrupar por Video e Minuto para calcular a média da temporada
min_counts = master_df.groupby(['video_id', 'minuto_arredondado']).size().reset_index(name='count')

# Filtrar range relevante
min_counts = min_counts[(min_counts['minuto_arredondado'] >= -10) & (min_counts['minuto_arredondado'] <= 130)]

plt.figure(figsize=(14, 6))
sns.lineplot(data=min_counts, x='minuto_arredondado', y='count', color='teal', errorbar='sd')
plt.title('Distribuição de Volume de Mensagens por Minuto (Média da Temporada)', fontsize=16)
plt.xlabel('Minuto Relativo ao Início (0 = Kickoff)')
plt.ylabel('Mensagens por Minuto')
plt.axvline(x=0, color='red', linestyle='--', alpha=0.5, label='Início 1ºT')
plt.axvline(x=60, color='blue', linestyle='--', alpha=0.5, label='Aprox. Início 2ºT')
plt.legend()
plt.show()

## 4. Análise por Período
Onde está o maior engajamento?

In [ ]:
plt.figure(figsize=(10, 6))
periodo_counts = master_df['periodo'].value_counts().sort_values(ascending=False)
sns.barplot(x=periodo_counts.index, y=periodo_counts.values, hue=periodo_counts.index, palette='viridis', legend=False)
plt.title('Total de Mensagens por Período do Jogo', fontsize=14)
plt.ylabel('Contagem Total')
plt.show()

## 5. Distribuição de Autores
Identificando o comportamento dos usuários.

In [ ]:
user_counts = master_df['autor'].value_counts()

plt.figure(figsize=(10, 5))
sns.histplot(user_counts, bins=50, kde=True, color='purple', log_scale=True)
plt.title('Distribuição de Mensagens por Autor (Escala Log)', fontsize=14)
plt.xlabel('Mensagens por Autor')
plt.ylabel('Frequência de Autores')
plt.show()

print("Top 10 Autores mais ativos (Super-fans):")
print(user_counts.head(10))

## 6. Heatmap de Atividade Sazonal
Cada linha representa um jogo diferente ao longo da temporada.

In [ ]:
pivot_table = master_df.pivot_table(index='video_id', columns='minuto_arredondado', values='mensagem', aggfunc='count').fillna(0)
pivot_table = pivot_table.loc[:, 0:110]

plt.figure(figsize=(16, 12))
sns.heatmap(pivot_table, cmap='rocket_r', cbar_kws={'label': 'Msgs/min'})
plt.title('Intensidade do Chat: Comparação entre Partidas da Temporada', fontsize=16)
plt.xlabel('Minuto de Jogo')
plt.ylabel('ID da Partida')
plt.show()

## 7. Tamanho das Mensagens
Distribuição do comprimento do texto.

In [ ]:
master_df['msg_len'] = master_df['mensagem'].astype(str).str.len()

plt.figure(figsize=(10, 5))
sns.kdeplot(data=master_df, x='msg_len', fill=True, color='orange')
plt.title('Distribuição do Comprimento das Mensagens', fontsize=14)
plt.xlabel('Número de Caracteres')
plt.ylabel('Densidade')
plt.xlim(0, 150)
plt.show()